# 第六章：CIFAR-10 图像分类实战

## 从 MLP 到 CNN——用 PyTorch 构建达到 85%+ 准确率的卷积网络

CIFAR-10 是计算机视觉的经典基准：32×32 彩色图片，10 个类别（飞机、汽车、鸟、猫、鹿、狗、蛙、马、船、卡车），
训练集 50,000 张，测试集 10,000 张。本章将完整演示从数据加载到模型调优的全流程。


## 6.1 CIFAR-10 数据集

### 数据标准化 (Normalization)

CIFAR-10 的 RGB 值范围是 [0, 255]。直接输入网络会导致梯度更新不稳定——大的像素值产生大的激活值，进而产生大的梯度。

标准化的公式：

$$

x_{	ext{norm}} = \frac{x - \mu}{\sigma}

$$

其中 $\mu$ 和 $\sigma$ 是整个训练集每个通道的均值和标准差。
归一化后每个通道的数据满足均值为 0、标准差为 1 的分布。这样做的三层好处：

1. **梯度稳定**：所有输入在同一尺度，梯度不会因为某层输入过大而爆炸
2. **收敛加速**：损失函数的等高线从狭长椭圆变成接近圆形，梯度下降路径更直接
3. **初始化兼容**：权重初始化假设输入均值为 0，归一化保证了这一点

CIFAR-10 的经验统计量（在 ImageNet 预训练时算的，已成为事实标准）：

$$

\mu = (0.4914, 0.4822, 0.4465), \quad \sigma = (0.2470, 0.2435, 0.2616)

$$

### 10 个类别

| 标签 | 类别 | 标签 | 类别 |
|------|------|------|------|
| 0 | airplane | 5 | dog |
| 1 | automobile | 6 | frog |
| 2 | bird | 7 | horse |
| 3 | cat | 8 | ship |
| 4 | deer | 9 | truck |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

# === 数据增强与标准化 ===
# CIFAR-10 的 RGB 均值和标准差
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# 训练数据增强
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # 随机裁剪（先 padding 到 40×40）
    transforms.RandomHorizontalFlip(),          # 随机水平翻转
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # 颜色抖动
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# 验证/测试：仅标准化
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# 加载数据集
train_dataset = datasets.CIFAR10(
    root='./data', train=True,
    download=True, transform=train_transform
)
test_dataset = datasets.CIFAR10(
    root='./data', train=False,
    download=True, transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"训练集: {len(train_dataset)} 张, 测试集: {len(test_dataset)} 张")


设备: cpu


  2%|▏         | 4.13M/170M [05:05<3:56:27, 11.7kB/s]

In [ ]:
# === 可视化数据集 ===
import numpy as np
import matplotlib.pyplot as plt
import torch
def imshow(img):
    """反标准化并显示图片"""
    img = img * torch.tensor(CIFAR10_STD).view(3, 1, 1) + torch.tensor(CIFAR10_MEAN).view(3, 1, 1)  # 从数据创建张量
    npimg = img.numpy()  # 转为 NumPy 数组
    plt.imshow(np.transpose(npimg, (1, 2, 0)))  # 显示图像
    plt.axis('off')

# 显示一个 batch 的前 16 张图
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(4, 4, figsize=(8, 8))  # 创建子图网格
for i, ax in enumerate(axes.flat):
    img = images[i] * torch.tensor(CIFAR10_STD).view(3, 1, 1) + torch.tensor(CIFAR10_MEAN).view(3, 1, 1)  # 从数据创建张量
    img = img.numpy().transpose(1, 2, 0)  # 转置两个维度
    img = np.clip(img, 0, 1)  # 限制数值范围，防止溢出
    ax.imshow(img)
    ax.set_title(classes[labels[i]], fontsize=9)
    ax.axis('off')
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/cifar10_samples.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 6.2 CNN 架构设计

### 为什么 MLP 不适合图像？

将 32×32×3 的图片展平为 3072 维向量会**丢失空间结构**——相邻像素之间的关系被破坏。
更重要的是，全连接层的参数量爆炸：第一层从 3072 到 512 就有 157 万个参数。

### CNN 的核心组件

| 层 | 操作 | 参数 | 作用 |
|----|------|------|------|
| **Conv2d** | 卷积 | kernel_size, stride, padding | 提取局部特征 |
| **BatchNorm2d** | 批归一化 | - | 稳定训练、加速收敛 |
| **ReLU** | 非线性激活 | - | 引入非线性 |
| **MaxPool2d** | 最大池化 | kernel_size | 降采样、增大感受野 |
| **Dropout** | 随机丢弃 | p | 正则化防止过拟合 |

### 卷积层的参数量

对于输入 $C_{in}$ 通道、输出 $C_{out}$ 通道、$K \times K$ 卷积核的 Conv2d：

$$

\text{Params} = C_{out} \times (C_{in} \times K^2 + 1)

$$

例如第一个卷积层 `Conv2d(3, 32, 3)`：参数 $= 32 \times (3 \times 9 + 1) = 896$——远少于全连接层。


### 6.2.1 卷积的数学本质

卷积不是凭空设计的——它来自信号处理中的**互相关 (Cross-Correlation)** 操作。

对于 2D 输入 $I$ 和卷积核 $K$：

$$
(I * K)(i, j) = \sum_{m} \sum_{n} I(i+m, j+n) \cdot K(m, n)
$$

这本质上是一个**滑动窗口内的加权求和**。每个输出位置只依赖于输入的一个局部区域——这就是**局部感受野 (Local Receptive Field)**。

卷积的参数量远小于全连接层，因为两个核心机制：

**1. 稀疏连接 (Sparse Connectivity)**

全连接层每个输出看所有输入。卷积层每个输出只看 $K \times K$ 个像素。同样从 3072 维输入到 32 维输出：全连接需 98,336 个参数，$3 \times 3$ 卷积只需 $3 \times 32 \times 9 + 32 = 896$ 个参数——差了 100 倍。

**2. 参数共享 (Parameter Sharing)**

同一个卷积核在图像的所有位置上滑动——无论"猫耳朵"出现在左上角还是右下角，同一个核都能检测到它。这就是**平移等变性 (Translation Equivariance)**：输入平移 → 输出同样平移。

**3. 特征层级 (Feature Hierarchy)**

浅层卷积检测低级特征（边缘、角点、颜色块），中层组合为中级特征（纹理、形状），深层抽象为高级语义（物体部件、完整物体）。这个层级结构不是人为设计的——它是从数据中学到的。

#### 卷积层 vs 全连接层的数学视角

全连接层：$y = \sigma(Wx + b)$，$W \in \mathbb{R}^{d_{out} \times d_{in}}$

卷积层可视为一个**稀疏的、权重共享的**全连接层——$W$ 中大部分元素为 0，且非零元素之间有循环模式：

$$
W_{\text{conv}} = \begin{bmatrix}
w_1 & w_2 & w_3 & 0 & 0 & \cdots \\
0 & w_1 & w_2 & w_3 & 0 & \cdots \\
0 & 0 & w_1 & w_2 & w_3 & \cdots \\
\vdots & & & \ddots & &
\end{bmatrix}
$$

这就是为什么卷积的参数量远小于全连接——不是因为"卷积"这个名字，而是因为稀疏连接 + 参数共享的约束。

#### 输出尺寸公式

$$H_{\text{out}} = \left\lfloor\frac{H_{\text{in}} - K + 2P}{S}\right\rfloor + 1$$

其中 $K$ 是卷积核大小，$P$ 是 padding，$S$ 是 stride。例如输入 32×32，$K=3$，$P=1$，$S=1$ → 输出仍是 32×32。


In [ ]:
# === 手工计算特征图尺寸 ===
def conv_out_size(in_size, kernel, stride=1, padding=0):
    """卷积输出尺寸: (W - K + 2P) / S + 1"""
    return (in_size - kernel + 2 * padding) // stride + 1  # 返回结果

# 追踪每层尺寸变化
print("输入: 32 × 32 × 3")
size = 32
size = conv_out_size(size, 3, padding=1)  # Conv(3→32, k=3, p=1)
print(f"Conv1 (p=1): {size} × {size} × 32")
size = conv_out_size(size, 2, stride=2)   # MaxPool(2×2, s=2)
print(f"MaxPool1: {size} × {size} × 32")

size = conv_out_size(size, 3, padding=1)  # Conv(32→64, k=3, p=1)
print(f"Conv2 (p=1): {size} × {size} × 64")
size = conv_out_size(size, 2, stride=2)   # MaxPool(2×2, s=2)
print(f"MaxPool2: {size} × {size} × 64")

size = conv_out_size(size, 3, padding=1)  # Conv(64→128, k=3, p=1)
print(f"Conv3 (p=1): {size} × {size} × 128")
size = conv_out_size(size, 2, stride=2)   # MaxPool(2×2, s=2)
print(f"MaxPool3: {size} × {size} × 128")

print(f"\n展平维度: {size}×{size}×128 = {size*size*128}")


### 6.2.2 批归一化 (BatchNorm)

对每个 mini-batch 的每个通道独立归一化：

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma\hat{x}_i + \beta$$

$\gamma,\beta$ 可学习——网络可以选择"不归一化"。训练用当前 batch 统计量，推理用全局移动平均。

**BatchNorm 解决的核心问题：**

1. **内部协变量偏移 (Internal Covariate Shift)**：网络每层输入分布随训练变化，迫使后续层不断适应。BatchNorm 将每层输入固定到均值为 0、方差为 1 的分布，消除偏移。
2. **梯度流改善**：归一化后激活值落在激活函数的线性区，梯度更容易传播——深层网络的训练速度提升 5-10 倍。
3. **对初始化的鲁棒性**：有 BatchNorm 后，权重初始化的精度要求大幅降低。

> 注意：BatchNorm 在 batch size 很小时（< 8）表现不稳定——此时 GroupNorm 或 LayerNorm 是更好的选择。


### 6.2.3 池化层 (Pooling)

池化层对特征图做降采样。最常见的 **Max Pooling** 在每个 $2 \times 2$ 窗口中取最大值：

$$y_{i,j} = \max_{m,n \in \{0,1\}} x_{2i+m, 2j+n}$$

#### 池化提供两个关键属性

**1. 平移不变性 (Translation Invariance)**

如果图像中的猫向右平移 2 个像素，Max Pooling 后的特征图可能完全不变——因为最大值仍然在那个窗口中。分类任务需要的是"有没有猫"，而不是"猫在哪个像素"。这种轻微的平移容忍度正是图像分类所需要的。

**2. 增大感受野**

每次 $2 \times 2$ 池化将特征图边长减半，下一层卷积的 $3 \times 3$ 核实际上覆盖了原图的 $6 \times 6$ 区域。经过 3 次池化后，顶层神经元可以看到原图的大部分区域——不需要更深的网络。

#### 池化的替代方案：Strided Convolution

Max Pooling 有一个根本局限：它**不可学习**——永远取最大值，网络无法决定保留什么。

**Strided Convolution**（步长 2 的 $3 \times 3$ 卷积）同时完成特征提取和降采样。因为卷积核可以学习，它能自己决定哪些信息应该保留——不像 Max Pooling 直接丢弃 75% 的信息。

现代主流架构（ResNet、EfficientNet）已全面用 Strided Conv 替换池化层。

#### Max Pooling vs Average Pooling vs Strided Conv

| 方法 | 操作 | 可学习 | 现代使用 |
|------|------|--------|---------|
| **Max Pooling** | 取窗口最大值 | 否 | 传统 CNN，逐渐被替代 |
| **Average Pooling** | 取窗口均值 | 否 | 全局平均池化（分类前） |
| **Strided Conv** | 步长 > 1 的卷积 | 是 | ResNet/EfficientNet 标配 |


In [ ]:
# === VGG 风格 CNN ===
import torch.nn as nn
class CIFAR10CNN(nn.Module):  # 所有网络层的基类
    """VGG 风格 CNN：卷积块 + 池化 → 全连接分类器"""
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        
        # Block 1: 32→16
        self.conv1 = nn.Sequential(  # 顺序容器
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(32),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(32),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32→16
            nn.Dropout2d(dropout * 0.5)
        )
        
        # Block 2: 16→8
        self.conv2 = nn.Sequential(  # 顺序容器
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(64),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(64),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16→8
            nn.Dropout2d(dropout * 0.7)
        )
        
        # Block 3: 8→4
        self.conv3 = nn.Sequential(  # 顺序容器
            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(128),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),  # 二维卷积层
            nn.BatchNorm2d(128),  # 二维批归一化
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 8→4
            nn.Dropout2d(dropout)
        )
        
        # 全连接分类器
        self.classifier = nn.Sequential(  # 顺序容器
            nn.Linear(128 * 4 * 4, 256),  # 全连接层 y=Wx+b
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),  # 随机丢弃神经元
            nn.Linear(256, num_classes)  # 全连接层 y=Wx+b
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.classifier(x)
        return x  # 返回结果

model = CIFAR10CNN().to(device)  # 将数据移到 GPU 或 CPU
total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数: {total_params:,}")
print(model)


## 6.3 训练策略

### 学习率调度

固定的学习率容易导致训练后期震荡或停滞。**余弦退火 (Cosine Annealing)** 是当前最常用的策略：

$$

\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T_{\max}}\pi\right)\right)

$$

### 训练技巧清单

- **Kaiming 初始化**：适配 ReLU 的权重初始化
- **BatchNorm**：加速收敛，允许更高学习率
- **数据增强**：RandomCrop + RandomFlip → 相当于增加 10 倍训练数据
- **Label Smoothing**：将 one-hot 目标从 `[0,0,1,0]` 平滑为 `[0.025,0.025,0.9,0.025]`
- **Gradient Clipping**：防止梯度爆炸


In [ ]:
# === 训练函数 ===
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    running_loss, correct, total = 0.0, 0, 0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()  # 反向传播，计算梯度
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()  # 用累积的梯度更新参数
        if scheduler:
            scheduler.step()  # 用梯度更新参数
        
        running_loss += loss.item()  # 取出单个数值
        _, predicted = outputs.max(1)  # 沿指定维度取最大值
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()  # 沿指定维度求和
        
        pbar.set_postfix({'loss': f'{running_loss/total*len(loader.dataset):.3f}',
                          'acc': f'{100.*correct/total:.1f}%'})
    
    return running_loss / len(loader), 100. * correct / total  # 返回结果

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)  # 将数据移到 GPU 或 CPU
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        running_loss += loss.item()  # 取出单个数值
        _, predicted = outputs.max(1)  # 沿指定维度取最大值
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()  # 沿指定维度求和
    
    return running_loss / len(loader), 100. * correct / total  # 返回结果

# === 设置 ===
model = CIFAR10CNN().to(device)  # 将数据移到 GPU 或 CPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# OneCycleLR：先升后降的学习率策略
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.1,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    pct_start=0.3  # 前 30% 步数用于 warmup
)

# 训练 20 epochs
num_epochs = 20
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Acc={train_acc:.2f}% | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.2f}%")

print(f"\n✅ 测试集最终准确率: {val_acc:.2f}%")


In [ ]:
# === 训练曲线 ===
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))  # 创建子图网格

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(True)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.axhline(y=85, color='green', linestyle='--', alpha=0.5, label='85% baseline')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curves'); ax2.legend(); ax2.grid(True)

plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/cifar10_training.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


## 6.4 模型评估与错误分析

准确率只是冰山一角。深入分析模型在哪些类别上出错，才能针对性改进。


In [ ]:
# === 混淆矩阵 ===
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    all_preds, all_targets = [], []
    for inputs, targets in loader:
        inputs = inputs.to(device)  # 将数据移到 GPU 或 CPU
        outputs = model(inputs)
        _, preds = outputs.max(1)  # 沿指定维度取最大值
        all_preds.extend(preds.cpu().numpy())  # 移到 CPU
        all_targets.extend(targets.numpy())  # 转为 NumPy 数组
    return np.array(all_preds), np.array(all_targets)  # 返回结果

y_pred, y_true = get_predictions(model, test_loader, device)

# 分类报告
print(classification_report(y_true, y_pred, target_names=classes))

# 混淆矩阵
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # 沿指定维度求和

plt.figure(figsize=(10, 8))  # 创建画布
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',  # 画热力图
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted'); plt.ylabel('True')  # 设置x轴标签
plt.title('CIFAR-10 Confusion Matrix (Normalized)')  # 设置图表标题
plt.tight_layout()  # 自动调整子图间距
plt.savefig('../fig/cifar10_confusion.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表


In [ ]:
# === 预测可视化：正确 vs 错误 ===
import numpy as np
import matplotlib.pyplot as plt
import torch
def show_predictions(model, loader, device, num=12):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    images_list, labels_list, preds_list, correct_list = [], [], [], []
    
    with torch.no_grad():  # 禁用梯度计算（推理/评估时用）
        for inputs, targets in loader:
            inputs_gpu = inputs.to(device)  # 将数据移到 GPU 或 CPU
            outputs = model(inputs_gpu)
            _, preds = outputs.max(1)  # 沿指定维度取最大值
            
            images_list.extend(inputs.cpu())  # 移到 CPU
            labels_list.extend(targets)
            preds_list.extend(preds.cpu())  # 移到 CPU
            correct_list.extend((preds.cpu() == targets).tolist())  # 移到 CPU
    
    # 找正确和错误的样本
    correct_idx = [i for i, c in enumerate(correct_list) if c][:num//2]
    wrong_idx = [i for i, c in enumerate(correct_list) if not c][:num//2]
    
    fig, axes = plt.subplots(4, 3, figsize=(12, 12))  # 创建子图网格
    for i, idx in enumerate(correct_idx + wrong_idx):
        ax = axes[i//3][i%3]
        img = images_list[idx]
        # 反标准化
        img = img * torch.tensor(CIFAR10_STD).view(3, 1, 1) + torch.tensor(CIFAR10_MEAN).view(3, 1, 1)  # 从数据创建张量
        img = img.numpy().transpose(1, 2, 0)  # 转置两个维度
        img = np.clip(img, 0, 1)  # 限制数值范围，防止溢出
        
        true_label = classes[labels_list[idx]]
        pred_label = classes[preds_list[idx]]
        color = 'green' if labels_list[idx] == preds_list[idx] else 'red'
        
        ax.imshow(img)
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=9)
        ax.axis('off')
    
    plt.tight_layout()  # 自动调整子图间距
    plt.savefig('../fig/cifar10_predictions.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
    plt.show()  # 显示图表

# 使用测试集显示（无数据增强）
test_loader_clean = DataLoader(
    datasets.CIFAR10(root='./data', train=False, download=False,
                     transform=transforms.Compose([
                         transforms.ToTensor(),
                         transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
                     ])),
    batch_size=128, shuffle=True
)
show_predictions(model, test_loader_clean, device)


## 6.5 迁移学习：站在巨人的肩膀上

从零训练 CNN 需要大量数据和算力。**迁移学习**使用在 ImageNet（140 万张图，1000 类）上预训练的模型，
仅微调最后几层即可在新任务上取得优异效果。

### 核心思想

ImageNet 预训练的卷积层已经学会了**通用视觉特征**（边缘 → 纹理 → 物体部件），
这些特征可迁移到 CIFAR-10。我们只需替换最后的分类头。

### 微调策略

1. 冻结 backbone（卷积层），仅训练分类头
2. 解冻最后几层，以极低学习率微调


In [ ]:
# === 迁移学习：ResNet-18 ===
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 加载预训练 ResNet-18
resnet = models.resnet18(weights='IMAGENET1K_V1')

# 冻结所有参数
for param in resnet.parameters():
    param.requires_grad = False

# 替换最后的全连接层
# ResNet-18 原始: fc(512 → 1000)
# CIFAR-10: fc(512 → 10)
resnet.fc = nn.Linear(512, 10)  # 全连接层 y=Wx+b

# 仅训练 fc 层
optimizer_transfer = optim.Adam(resnet.fc.parameters(), lr=0.001)
resnet = resnet.to(device)  # 将数据移到 GPU 或 CPU

# 需要调整输入尺寸：ResNet 期望 224×224，CIFAR-10 是 32×32
# 方法：在 transform 中加 Resize(224)
transfer_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

transfer_dataset = datasets.CIFAR10(
    root='./data', train=True,
    download=False, transform=transfer_transform
)
transfer_loader = DataLoader(transfer_dataset, batch_size=64, shuffle=True)

transfer_test = datasets.CIFAR10(
    root='./data', train=False,
    download=False, transform=transfer_transform
)
transfer_test_loader = DataLoader(transfer_test, batch_size=128, shuffle=False)

# 快速微调 3 epochs 演示
for epoch in range(3):
    train_loss, train_acc = train_epoch(
        resnet, transfer_loader, nn.CrossEntropyLoss(),
        optimizer_transfer, None, device
    )
    val_loss, val_acc = evaluate(resnet, transfer_test_loader, nn.CrossEntropyLoss(), device)
    print(f"Epoch {epoch+1}: Val Acc={val_acc:.2f}%")

print(f"\n迁移学习 ResNet-18 准确率: {val_acc:.2f}% (仅训练 fc 层 3 epochs)")


## 本章小结

| 组件 | 要点 |
|------|------|
| CIFAR-10 | 32×32 RGB，10 类，50K 训练 |
| CNN 架构 | Conv → BN → ReLU → Pool 堆叠 + FC 分类头 |
| 数据增强 | RandomCrop + RandomFlip + ColorJitter |
| 学习率 | OneCycleLR / CosineAnnealing |
| 正则化 | Dropout + Weight Decay + BatchNorm |
| 评估 | 混淆矩阵 + 分类报告 + 错误样本可视化 |
| 迁移学习 | ImageNet 预训练 → 替换分类头 → 微调 |

✅ 第六章完成！CNN 从数据到训练到评估的完整闭环。
